In [ ]:
import torch
import torch.nn.functional as F
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm

from types import SimpleNamespace
from torch.utils.data import TensorDataset, DataLoader, random_split
from dataset_loader import NASDatasetFactory,load_nas201_api,arch_to_tensor


from models.flow import FlowNet
from models.nas201_models import VAE_dist,vae_accuracy_loss
from models.nas301_models import VAE_nas301

from torch.utils.data import DataLoader
from train import pretrain_and_freeze_vae,run_training
from utils_functions.utils import build_accuracy_pairs, generate_archs,decoded_x_to_nas201_arch, query_nas201_accuracy,set_seed
from utils_functions.plots_utils import compare_accuracy_distributions

tensorflow is not installed.


In [ ]:
import pandas as pd

def load_csv_as_dataset(csv_path):
    df = pd.read_csv(csv_path)
    feature_cols = [col for col in df.columns if col.startswith("x_")]

    feature_cols = sorted(
        feature_cols,
        key=lambda c: int(c.split("_")[1])
    )

    X = df[feature_cols].values
    Y = df["accuracy"].values

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    dataset = TensorDataset(X, Y)

    print("CSV caricato:", csv_path)
    print("Numero esempi:", len(dataset))
    return X, Y, dataset

X, Y, dataset_301 = load_csv_as_dataset(
    "datasets/nas301/nas301_dataset.csv")

CSV caricato: datasets/nas301/nas301_dataset.csv
Numero esempi: 50000


In [ ]:
train_size = int(0.8 * len(dataset_301))
test_size = len(dataset_301) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(
    dataset_301,
    [train_size, test_size],
    generator=generator
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 40000
Test size: 10000


In [ ]:
args = SimpleNamespace(
    benchmark_name="NAS201",
    n_samples=1000,          # piccolo per test veloce
    performance_model=None,  # lo carica la factory se return_model=True
    outer_epochs=20,
    N=1024,
    latent_dim=32,
    batch_size=64,

    vae_epochs=5,
    pretrain_vae_epochs=200,
    pretrain_batch_size=64,
    pretrain_fraction=0.5,

    flow_epochs=5,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.2,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_mane = "nas301",
    dataset_name="cifar10",
    nas_hp=None,
    nas_metric="surrogate_accuracy",

    train_dataset=train_dataset,
    test_dataset=test_dataset,
    pos_weight_value = 5,
    weight_sharing = False,
    min_delta = 0.001,

    seed=42,
    device="cuda",
)

history, model_VAE, flow, test_dataset, api_or_model = run_training(args)

Dataset NAS201 già estratto.
Architetture NAS201 totali: 15625
Dataset available

 PRETRAIN VAE


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x504 and 80x128)

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    benchmark_name="NAS301",

    n_samples=1000,
    performance_model=None,

    outer_epochs=20,
    N=1024,
    latent_dim=32,
    batch_size=64,

    vae_epochs=5,
    pretrain_vae_epochs=200,
    pretrain_batch_size=64,
    pretrain_fraction=0.5,

    flow_epochs=5,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.2,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_mane="nas301",
    dataset_name="cifar10",
    nas_hp=None,
    nas_metric="surrogate_accuracy",

    train_dataset=train_dataset,
    test_dataset=test_dataset,
    weight_sharing = False,

    pos_weight_value=5,
    min_delta = 0.001,
    seed=42,
    device="cuda",
)

In [ ]:
from nas301_flow_vs_random_20_runs import run_flow_vs_random_experiment

results_df, samples_df, summary_df = run_flow_vs_random_experiment(
    base_args=args,
    run_training_fn=run_training,
    n_runs=20,
    max_test_samples=None,
    initial_seed=42,
    output_dir="results_flow_vs_random"
)

# Esperimenti con ws


In [1]:
import torch
import torch.nn.functional as F
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm

from types import SimpleNamespace
from torch.utils.data import TensorDataset, DataLoader, random_split
from dataset_loader import NASDatasetFactory,load_nas201_api,arch_to_tensor


from models.flow import FlowNet
from models.nas201_models import VAE_dist,vae_accuracy_loss
from models.nas301_models import VAE_nas301

from torch.utils.data import DataLoader
from train import pretrain_and_freeze_vae,run_training
from utils_functions.utils import build_accuracy_pairs, generate_archs,decoded_x_to_nas201_arch, query_nas201_accuracy,set_seed
from utils_functions.plots_utils import compare_accuracy_distributions

tensorflow is not installed.


In [2]:
from types import SimpleNamespace

args = SimpleNamespace(
    benchmark_name="NAS201",
    n_samples=1000,          # piccolo per test veloce
    performance_model=None,  # lo carica la factory se return_model=True
    outer_epochs=20,
    N=1024,
    latent_dim=32,
    batch_size=64,

    vae_epochs=5,
    pretrain_vae_epochs=200,
    pretrain_batch_size=64,
    pretrain_fraction=0.5,

    flow_epochs=5,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.2,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_mane = "nas301",
    dataset_name="cifar10",
    nas_hp=None,
    nas_metric="surrogate_accuracy",

    train_dataset=None,
    test_dataset=None,
    pos_weight_value = 5,
    weight_sharing = True,
    min_delta = 0.001,

    seed=42,
    device="cuda",
)

history, model_VAE, flow, test_dataset, api_or_model = run_training(args)

Dataset NAS201 già estratto.
Architetture NAS201 totali: 15625

PRETRAIN VAE
VAE pretrain epoch 000 | loss=1.350278 | recon=1.350278 | kl=0.007190 | acc_loss=0.000000
VAE pretrain epoch 050 | loss=0.003243 | recon=0.003243 | kl=24.727674 | acc_loss=0.000000
VAE pretrain epoch 100 | loss=0.000144 | recon=0.000144 | kl=36.149041 | acc_loss=0.000000
Early stopping: loss below threshold at epoch 113, loss=0.000100
VAE pretrained and frozen.

 OUTER EPOCH 1/20 ==========
Evaluating networks...
Epoch   0 | Batch    0 | Loss (M=4): 2.3784 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch   25 | Loss (M=4): 2.2770 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch   50 | Loss (M=4): 2.2284 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch   75 | Loss (M=4): 2.1636 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch  100 | Loss (M=4): 2.1415 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch  125 | Loss (M=4): 2.1127 | LR: 0.025000 | Path noti: 969
Epoch   0 | Batch  150 | Loss (M=4): 2.1610 | LR: 

KeyboardInterrupt: 